# multi-task fc 实验可视化

处理 `record/4-LIF_fc/`、`record/5-LIF_fc/`、`record/5-LIF_fc-dropout/` 下的日志：
- 柱状图：每种拓扑最后一个 epoch 的 acc1, acc2，按 seed 求平均（三组各一张，每张 2 子图）
- 折线图：每种拓扑每 epoch 的 loss，按 seed 求平均（无误差线）

In [ ]:
import os
import re
import json
from collections import defaultdict
import numpy as np
import matplotlib.pyplot as plt

RECORD_DIR = os.path.join(os.path.dirname(os.path.abspath('__file__') if '__file__' in dir() else '.'), 'record')
GROUPS = ['4-LIF_fc', '5-LIF_fc', '5-LIF_fc-dropout', 'fc']

EPOCH_RE = re.compile(r'Epoch:\s*(\d+)\s+Loss:\s*([-\d.]+)\s+acc1:\s*([-\d.]+)\s+acc2:\s*([-\d.]+)')
LOG_NAME_RE = re.compile(r'^(?P<model>.+?)_seed(?P<seed>\d+)\.log$')

In [ ]:
def parse_log(path):
    rows = []
    with open(path, 'r') as f:
        for line in f:
            m = EPOCH_RE.search(line)
            if m:
                rows.append((int(m.group(1)), float(m.group(2)),
                             float(m.group(3)), float(m.group(4))))
    return rows


def load_group(group):
    group_dir = os.path.join(RECORD_DIR, group)
    data = defaultdict(dict)  # model -> seed -> rows
    for fname in sorted(os.listdir(group_dir)):
        m = LOG_NAME_RE.match(fname)
        if not m:
            continue
        model = m.group('model')
        seed = int(m.group('seed'))
        data[model][seed] = parse_log(os.path.join(group_dir, fname))
    return data

In [ ]:
data = {group: load_group(group) for group in GROUPS}
# HH baseline lives in record/fc/ as HH_fc_seed*.log; alias HH_fc → HH so the chart can use 'HH'
if 'fc' in data and 'HH_fc' in data['fc']:
    data['fc']['HH'] = data['fc'].pop('HH_fc')
for group in GROUPS:
    print(f'{group}: {len(data[group])} models')
    for m, seeds in data[group].items():
        n_seeds = len(seeds)
        n_epochs = len(next(iter(seeds.values())))
        print(f'  {m}: {n_seeds} seeds, {n_epochs} epochs')

In [ ]:
total = 0
for i in [1, 2, 3]:
    total += data['fc']['HH'][i][-1][1]
print(total / 3)

## 柱状图：最后一个 epoch 的 acc1 / acc2（按 seed 平均）

In [ ]:
def last_epoch_accs(model_seeds):
    acc1, acc2 = [], []
    for rows in model_seeds.values():
        if rows:
            _, _, a1, a2 = rows[-1]
            acc1.append(a1); acc2.append(a2)
    return np.mean(acc1), np.mean(acc2), np.std(acc1), np.std(acc2)

for group in GROUPS:
    models = sorted(data[group].keys())
    acc1_means, acc2_means, acc1_stds, acc2_stds = [], [], [], []
    for m in models:
        a1, a2, s1, s2 = last_epoch_accs(data[group][m])
        acc1_means.append(a1); acc2_means.append(a2)
        acc1_stds.append(s1); acc2_stds.append(s2)
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    x = np.arange(len(models))

    # 左图 acc1
    axes[0].bar(x, acc1_means, yerr=acc1_stds, capsize=4, color='steelblue', alpha=0.8)
    axes[0].set_xticks(x); axes[0].set_xticklabels(models, rotation=45, ha='right')
    axes[0].set_ylabel('acc1 (%)')
    axes[0].set_title(f'{group} - acc1 (task 1, last epoch)')
    axes[0].grid(axis='y', alpha=0.3)

    # 在每根柱子顶部标均值
    for i, m in enumerate(acc1_means):
        axes[0].text(i, m - 10, f'{m:.2f}', ha='center', va='bottom', fontsize=9)

    # 右图 acc2（同样处理）
    axes[1].bar(x, acc2_means, yerr=acc2_stds, capsize=4, color='indianred', alpha=0.8)
    axes[1].set_xticks(x); axes[1].set_xticklabels(models, rotation=45, ha='right')
    axes[1].set_ylabel('acc2 (%)')
    axes[1].set_title(f'{group} - acc2 (task 2, last epoch)')
    axes[1].grid(axis='y', alpha=0.3)

    for i, m in enumerate(acc2_means):
        axes[1].text(i, m - 10, f'{m:.2f}', ha='center', va='bottom', fontsize=9)

    fig.tight_layout()
    plt.show()

## 配对柱状图：HH 基线 + (4-LIF, 5-LIF-dropout, 5-LIF) 配对

按拓扑对应关系排开：HH 在最左侧作为基准，然后每个 5-LIF 拓扑组内 3 根柱子 (4-LIF baseline / 5-LIF dropout / 5-LIF full)。

In [ ]:
# Each group: (short label, (4-LIF model, 5-LIF model))
# Label = 4-LIF structure written as primaries+mixings (e.g. "1+3"). The 5-LIF topology is
# uniquely identified as "1+" + the 4-LIF structure, so the 4-LIF label suffices.
# HH baseline comes from record/fc/ (HH_fc_seed*.log) and is loaded as data['fc']['HH'].
PLOT_GROUPS = [
    ('HH',         None),
    ('1+3',        ('LIF_1_3',       'LIF_1_3_1')),
    ('2+2',        ('LIF_2_2',       'LIF_1_2_2')),
    ('1+2+1',      ('LIF_1_2_1',     'LIF_1_1_2_1')),
    ('1+1+1+1',    ('LIF_1_1_1_1',   'LIF_1_1_1_1_1')),
    ('3+1',        ('LIF_hh_fc',     'LIF_1_3_1')),
    ('1+4',        ('4LIF_fc',       'LIF_1_4')),
]

def fetch_acc(group, model, task):
    """Return (mean, std) of last-epoch acc for given group/model/task across seeds."""
    if model not in data[group]:
        return None, None
    rows_dict = data[group][model]
    vals = [r[-1][2 + task] for r in rows_dict.values() if r]  # task=0 → acc1, task=1 → acc2
    if not vals:
        return None, None
    return np.mean(vals), np.std(vals)

# Layout: HH at position 0, then 3 bars per group, with gaps between groups
bar_w = 0.25
xtick_labels = []

x_cursor = 0.0
for label, pair in PLOT_GROUPS:
    if pair is None:
        xtick_labels.append(label)
        x_cursor += bar_w * 4
    else:
        xtick_labels.append(label)
        x_cursor += bar_w * 5

# Plot
fig, axes = plt.subplots(1, 2, figsize=(18, 5))

# For each task, rebuild bar_specs with the corresponding mean/std
def build_bars(task):
    bars = []
    for label, pair in PLOT_GROUPS:
        if pair is None:
            m, s = fetch_acc('fc', 'HH', task)
            bars.append((0.0, m if m is not None else 0.0, s if s is not None else 0.0, 'gray', 'HH'))
        else:
            m4, m5 = pair
            for dx, color, kind in zip([-bar_w, 0.0, bar_w],
                                       ['steelblue', 'indianred', 'mediumseagreen'],
                                       ['4-LIF', '5-LIF\n(dropout)', '5-LIF\n(full)']):
                if kind == '4-LIF':
                    m, s = fetch_acc('4-LIF_fc', m4, task)
                elif kind == '5-LIF\n(dropout)':
                    m, s = fetch_acc('5-LIF_fc-dropout', m5, task)
                else:
                    m, s = fetch_acc('5-LIF_fc', m5, task)
                bars.append((0.0, m if m is not None else 0.0, s if s is not None else 0.0, color, kind))
    return bars

# Recompute x positions for each task
def recompute_x():
    x_cursor = 0.0
    out = []
    for label, pair in PLOT_GROUPS:
        if pair is None:
            out.append(x_cursor)
            x_cursor += bar_w * 4
        else:
            for dx in [-bar_w, 0.0, bar_w]:
                out.append(x_cursor + dx + bar_w)
            x_cursor += bar_w * 5
    return out

x_positions = recompute_x()
for task_idx, task_name in enumerate(['acc1', 'acc2']):
    ax = axes[task_idx]
    bars = build_bars(task_idx)
    # Build xtick positions per group (group center)
    group_centers_list = []
    cursor = 0.0
    for label, pair in PLOT_GROUPS:
        if pair is None:
            group_centers_list.append(cursor)
            cursor += bar_w * 4
        else:
            group_centers_list.append(cursor + bar_w)
            cursor += bar_w * 5
    for i, ((x, m, s, color, kind), xp) in enumerate(zip(bars, x_positions)):
        ax.bar(xp, m, yerr=s, capsize=3, color=color, alpha=0.85, width=bar_w*0.95)
        if m > 0:
            ax.text(xp, m + 2.5, f'{m:.1f}', ha='center', va='bottom', fontsize=7)
    ax.set_xticks(group_centers_list)
    ax.set_xticklabels(xtick_labels, fontsize=10, rotation=0, ha='center')
    ax.set_ylabel(f'{task_name} (%)')
    ax.set_title(f'{task_name} (mean over seeds, last epoch)')
    ax.grid(axis='y', alpha=0.3)
    # Legend (manually)
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor='gray',         alpha=0.85, label='HH (baseline)'),
        Patch(facecolor='steelblue',    alpha=0.85, label='4-LIF'),
        Patch(facecolor='indianred',    alpha=0.85, label='5-LIF (dropout)'),
        Patch(facecolor='mediumseagreen', alpha=0.85, label='5-LIF (full)'),
    ]
    ax.legend(handles=legend_elements, loc='lower left', fontsize=8)
    # y-axis range: ensure all bars visible
    ymax = max((m + (s if s else 0)) for _, m, s, _, _ in bars) + 2
    ax.set_ylim(0, max(ymax, 100))

fig.suptitle('HH baseline + 4-LIF / 5-LIF-dropout / 5-LIF pairs (multi-task FC, last epoch)', fontsize=12)
fig.tight_layout()
plt.show()

## 折线图：每 epoch loss（按 seed 平均，无误差线）

In [ ]:
def per_epoch_loss_mean(model_seeds):
    """Return (epochs, mean_loss_per_epoch) averaged across seeds."""
    epoch_to_losses = defaultdict(list)
    for rows in model_seeds.values():
        for ep, loss, _, _ in rows:
            epoch_to_losses[ep].append(loss)
    epochs = sorted(epoch_to_losses)
    means = [np.mean(epoch_to_losses[e]) for e in epochs]
    return np.array(epochs), np.array(means)

for group in GROUPS:
    fig, ax = plt.subplots(figsize=(9, 5))
    for m in sorted(data[group].keys()):
        epochs, means = per_epoch_loss_mean(data[group][m])
        ax.plot(epochs, means, marker='o', markersize=3, label=m)
    ax.set_xlabel('Epoch'); ax.set_ylabel('Loss (mean over seeds)')
    ax.set_title(f'{group} - validation loss across epochs (mean over seeds)')
    ax.legend(loc='best', fontsize=8); ax.grid(alpha=0.3)
    fig.tight_layout(); plt.show()